# HIT140 Assessment 2: Defender age and fouls at the FIFA World Cup 2026

**Analytic question:** Among defenders who played at the FIFA World Cup 2026, do older defenders commit fewer fouls per 90 minutes than younger defenders? Only defenders with at least N minutes on the field are included.

## Data Wrangling

### Data Acquisition

Player statistics are taken from FBref. Two tables for the 2026 World Cup were exported using the CSV export feature of the site.

| File | Table Name | Source |
|---|---|---|
| `data/raw/fbref_player_standard.csv` | Player Standard Stats (position, age, minutes, cards) | https://fbref.com/en/comps/1/stats/World-Cup-Stats |
| `data/raw/fbref_player_misc.csv` | Player Miscellaneous Stats (fouls, cards, tackles) | https://fbref.com/en/comps/1/misc/World-Cup-Stats |

### Cleaning

Opening the file in a text editor shows the table starts with the labels on the 6th row, so we skip the first 5 rows to get the data only

```
--- When using SR data, please cite us and provide a link and/or a mention.


 
,,,,,,,Performance,Performance,Performance,Performance,Performance,Performance,Performance,Performance,Performance,Performance,Performance,Performance,,-additional
Rk,Player,Pos,Squad,Age,Born,90s,CrdY,CrdR,2CrdY,Fls,Fld,Off,Crs,Int,TklW,PKwon,PKcon,OG,Matches,-9999
1,Brenden Aaronson,MF,us USA,25,2000,0.8,0,0,0,2,1,0,1,0,1,,,0,Matches,5bc43860
2,Thelo Aasgaard,MF,no Norway,24,2002,1.0,0,0,0,0,3,0,0,3,1,,,0,Matches,c88a28b9
...
```

In [1]:
import pandas as pd

standard = pd.read_csv("data/raw/fbref_player_standard.csv", skiprows=5)
misc = pd.read_csv("data/raw/fbref_player_misc.csv", skiprows=5)

print("standard:", standard.shape)
print("misc:", misc.shape)

standard.head()

standard: (1040, 26)
misc: (1039, 21)

,Rk,Player,Pos,Squad,Age,Club,Born,MP,Starts,Min,...,PKatt,CrdY,CrdR,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK,Matches,-9999
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-9999
1,1.0,Brenden Aaronson,MF,us USA,25.0,1.eng Leeds United,2000.0,1.0,1.0,76.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Matches,5bc43860
2,2.0,Thelo Aasgaard,MF,no Norway,24.0,1.sct Rangers,2002.0,1.0,1.0,90.0,...,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,Matches,c88a28b9
3,3.0,Hamza Abdelkarim,FW,eg Egypt,18.0,4.es Barcelona B,2008.0,4.0,0.0,60.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Matches,057d8f12
4,4.0,Hossam Abdelmaguid,DF,eg Egypt,25.0,1.eg Zamalek SC,2001.0,2.0,0.0,59.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Matches,accbfc94


In [2]:
# Check which columns we have
print("columns in standard:", list(standard.columns))
print()
print("columns in misc:", list(misc.columns))

columns in standard: ['Rk', 'Player', 'Pos', 'Squad', 'Age', 'Club', 'Born', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY', 'CrdR', 'Gls.1', 'Ast.1', 'G+A.1', 'G-PK.1', 'G+A-PK', 'Matches', '-9999']

columns in misc: ['Rk', 'Player', 'Pos', 'Squad', 'Age', 'Born', '90s', 'CrdY', 'CrdR', '2CrdY', 'Fls', 'Fld', 'Off', 'Crs', 'Int', 'TklW', 'PKwon', 'PKcon', 'OG', 'Matches', '-9999']

Columns kept for this question (names as FBref abbreviates them, with FBref's glossary meaning where it isn't obvious):

| Table | Column | Meaning |
|---|---|---|
| both | `player_id` | FBref's player identifier (originally labelled -9999) for join |
| standard | `Player` | Player name |
| standard | `Pos` | Position: GK, DF, MF, FW, two positions per player based on time in role |
| standard | `Squad` | National team |
| standard | `Age` | Age in years at the time of export |
| standard | `MP` | Matches played |
| standard | `Starts` | Matches started |
| standard | `Min` | Minutes played |
| standard | `90s` | Minutes played divided by 90 (number of full matches' worth of playing time) |
| misc | `Fls` | Fouls committed |
| misc | `Fld` | Fouls drawn (fouls suffered) |
| misc | `CrdY` | Yellow cards |
| misc | `CrdR` | Red cards |
| misc | `2CrdY` | Second yellow card in a match (counted as a red) |

Everything else is dropped: the row number `Rk`, the `Matches` link column, club, goals and assists, the per-90 goal columns, offsides, crosses, interceptions, tackles won, own goals, and the penalty columns (`PKwon`, `PKcon`), which are empty for every player in this export.

In [3]:
# Remove the NaN junk row
standard = standard.dropna(subset=["Player"])
# And reinfer types after dropping junk rows
standard = standard.convert_dtypes()

# Label the player id row, which is labelled -9999 in the raw data
standard = standard.rename(columns={"-9999": "player_id"})
misc = misc.rename(columns={"-9999": "player_id"})

# select columns useful for this question
standard = standard[["player_id", "Player", "Pos", "Squad", "Age", "MP", "Starts", "Min", "90s"]]
misc = misc[["player_id", "Fls", "Fld", "CrdY", "CrdR", "2CrdY"]]

# remove the country code from Squad
standard["Squad"] = standard["Squad"].str.split(" ", n=1).str[1]

print("standard:", standard.shape)
print("misc:", misc.shape)
standard.head()

standard: (1039, 9)
misc: (1039, 6)

,player_id,Player,Pos,Squad,Age,MP,Starts,Min,90s
1,5bc43860,Brenden Aaronson,MF,USA,25,1,1,76,0.8
2,c88a28b9,Thelo Aasgaard,MF,Norway,24,1,1,90,1.0
3,057d8f12,Hamza Abdelkarim,FW,Egypt,18,4,0,60,0.7
4,accbfc94,Hossam Abdelmaguid,DF,Egypt,25,2,0,59,0.7
5,bff15d74,Mohamed Abdelmonem,DF,Egypt,27,2,1,14,0.2


### Structuring

Fouls data (in `misc`) and player details (in `standard`) are in separate dataframes.

Each dataframe has one row per player. Each player is uniquely identified by the `player_id` column, so we do an inner join on `player_id`. We expect this to result in a dataframe with the same number of rows (1039) and 14 columns (sum of all columns in each dataframe, minus 1 because player_id will only appear once) 

Validate the structure so we know if anything breaks our assumptions.
- check the output shape is (1039, 14)
- use `validate="one_to_one"` to check that there are no duplicate player_ids within either input dataframe

In [4]:
players = standard.merge(misc, on="player_id", how="inner", validate="one_to_one")

print("players:", players.shape)
assert players.shape == (1039, 14), "join did not produce the expected shape"

players.head()

players: (1039, 14)

,player_id,Player,Pos,Squad,Age,MP,Starts,Min,90s,Fls,Fld,CrdY,CrdR,2CrdY
0,5bc43860,Brenden Aaronson,MF,USA,25,1,1,76,0.8,2,1,0,0,0
1,c88a28b9,Thelo Aasgaard,MF,Norway,24,1,1,90,1.0,0,3,0,0,0
2,057d8f12,Hamza Abdelkarim,FW,Egypt,18,4,0,60,0.7,2,0,0,0,0
3,accbfc94,Hossam Abdelmaguid,DF,Egypt,25,2,0,59,0.7,1,0,0,0,0
4,bff15d74,Mohamed Abdelmonem,DF,Egypt,27,2,1,14,0.2,1,0,0,0,0


### Filtering

The question is about defenders, so we can exclude players who are not defenders.

FBref lists players' positions as `DF`, `MF`, `FW` or `GK`, or two of these where a player played in both roles.

Only players listed as `DF` only are kept. The foul data in the `misc` table is recorded per-player, and so for players who were in multiple roles we cannot disaggregate the foul data to determine which of two possible positions a player was in when a foul was awarded.

In [5]:
# select only pure "DF" players
defenders = players[players["Pos"] == "DF"]

print("defenders:", defenders.shape)

defenders: (306, 14)

### Feature Construction

Comparing players on raw foul count doesn't work- as playing time has significant variation and impacts the number of fouls comitted. To address this, we treat the outcome variable as a rate: fouls per 90 minutes played. FBref's `90s` column represents the total number of minutes played divided by 90.

$$\text{fouls per 90} = \frac{\text{Fls}}{\text{90s}}$$

In [6]:
defenders["fouls_per_90"] = defenders["Fls"] / defenders["90s"]

defenders[["Player", "Squad", "Age", "90s", "Fls", "fouls_per_90"]].head(10)

,Player,Squad,Age,90s,Fls,fouls_per_90
3,Hossam Abdelmaguid,Egypt,25,0.7,1,1.428571
4,Mohamed Abdelmonem,Egypt,27,0.2,1,5.0
5,Ali Abdi,Tunisia,32,3.0,4,1.333333
6,Saud Abdulhamid,Saudi Arabia,26,3.0,4,1.333333
7,Abdulla Abdullaev,Uzbekistan,28,2.0,5,2.5
9,Husam Abu Dahab,Jordan,26,2.0,2,1.0
10,Mohammad Abu Hasheesh,Jordan,31,0.1,0,0.0
11,Mohannad Abu Taha,Jordan,23,2.9,2,0.689655
12,Mo Abualnadi,Jordan,25,0.8,0,0.0
19,Ricardo Adé,Haiti,36,3.0,6,2.0


### Interest tangent: who's the most foul player of them all (per 90)

In [7]:
defenders[["Player", "Squad", "Age", "90s", "Fls", "fouls_per_90"]].sort_values("fouls_per_90", ascending=False)

,Player,Squad,Age,90s,Fls,fouls_per_90
760,Anthony Ralston,Scotland,27,0.1,2,20.0
810,Mustafa Saadoon,Iraq,25,0.2,2,10.0
740,Jackson Porozo,Ecuador,25,0.3,2,6.666667
4,Mohamed Abdelmonem,Egypt,27,0.2,1,5.0
681,Salim Obaid,Jordan,34,0.2,1,5.0
...,...,...,...,...,...,...
606,Lucas Mendes,Qatar,35,0.0,0,<NA>
617,Yerry Mina,Colombia,31,0.0,0,<NA>
747,Marc Pubill,Spain,22,0.0,0,<NA>
822,Shurandy Sambo,Curaçao,24,0.0,0,<NA>


Two things came out of this tangent that we can fix:

Some players have a `90s` value of 0. They did play (FBref only lists players who played), but for fewer than five minutes, which results in `90s` rounding to 0 (4/90 = 0.04, rounds down). We can address this by using the `Mins` column instead of `90s`. The `fouls_per_90` column is then calculated as `Fls * 90 / Min`.

The max value of 20 for Anthony Ralston is absurdly high. His 90s value is 0.1, or about nine minutes. It's hard to say from this that Ralston is a player who is particularly likely to commit fouls, because there is too little playing time for the rate to be reliable.

In [8]:
# calculate the rate from the minutes instead of rounded 90s column
defenders["fouls_per_90"] = defenders["Fls"] * 90 / defenders["Min"]

defenders[["Player", "Squad", "Age", "Min", "Fls", "fouls_per_90"]].sort_values("fouls_per_90", ascending=False).head(10)

,Player,Squad,Age,Min,Fls,fouls_per_90
760,Anthony Ralston,Scotland,27,11,2,16.363636
810,Mustafa Saadoon,Iraq,25,18,2,10.0
4,Mohamed Abdelmonem,Egypt,27,14,1,6.428571
740,Jackson Porozo,Ecuador,25,29,2,6.206897
681,Salim Obaid,Jordan,34,21,1,4.285714
614,Thomas Meunier,Belgium,34,190,9,4.263158
907,Junnosuke Suzuki,Japan,22,43,2,4.186047
476,Bekhruz Karimov,Uzbekistan,18,179,8,4.022346
425,Roger Ibanez,Brazil,27,45,2,4.0
561,Mohamed Manai,Qatar,23,69,3,3.913043
